## Feature generailization

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif

def preprocess_with_interactions_selection(
    df,
    target_col="Churn",
    id_cols=None,
    test_size=0.2,
    random_state=42,
    k_features=50  # number of features to keep
):
    df = df.copy()

    # -------------------------
    # 1. Drop ID columns
    # -------------------------
    if id_cols:
        df = df.drop(columns=id_cols, errors="ignore")

    # -------------------------
    # 2. Split target
    # -------------------------
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=test_size, random_state=random_state
    )

    # -------------------------
    # 3. Identify column types
    # -------------------------
    numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
    categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns

    # -------------------------
    # 4. Pipelines
    # -------------------------

    # Numeric: impute → interactions → scale
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("poly", PolynomialFeatures(
            degree=2,
            interaction_only=True,
            include_bias=False
        )),
        ("scaler", StandardScaler())
    ])

    # Categorical: impute → one-hot
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    # Combine
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])

    preprocessor.set_output(transform="pandas")

    # -------------------------
    # 5. Feature Selection (keeps names!)
    # -------------------------
    selector = SelectKBest(score_func=f_classif, k=k_features)

    full_pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("feature_selection", selector)
    ])

    # -------------------------
    # 6. Transform
    # -------------------------
    X_train_processed = full_pipeline.fit_transform(X_train, y_train)
    X_test_processed = full_pipeline.transform(X_test)

    # -------------------------
    # 7. Get selected feature names
    # -------------------------
    feature_names = full_pipeline.named_steps["preprocessing"].get_feature_names_out()
    selected_mask = full_pipeline.named_steps["feature_selection"].get_support()

    selected_features = feature_names[selected_mask]

    # Convert to DataFrame with names
    X_train_processed = pd.DataFrame(X_train_processed, columns=selected_features)
    X_test_processed = pd.DataFrame(X_test_processed, columns=selected_features)

    return (
        X_train_processed,
        X_test_processed,
        y_train.reset_index(drop=True),
        y_test.reset_index(drop=True),
        full_pipeline,
        selected_features
    )

In [2]:
df = pd.read_csv("E Commerce Dataset.csv")

X_train, X_test, y_train, y_test, pipeline, selected_features = \
    preprocess_with_interactions_selection(
        df,
        target_col="Churn",         # change if different
        id_cols=["CustomerID"],     # optional
        test_size=0.2,
        random_state=42,
        k_features=50               # number of features to keep
    )

In [3]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4504 entries, 0 to 4503
Data columns (total 50 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   num__Tenure                                         4504 non-null   float64
 1   num__NumberOfDeviceRegistered                       4504 non-null   float64
 2   num__SatisfactionScore                              4504 non-null   float64
 3   num__Complain                                       4504 non-null   float64
 4   num__DaySinceLastOrder                              4504 non-null   float64
 5   num__CashbackAmount                                 4504 non-null   float64
 6   num__Tenure CityTier                                4504 non-null   float64
 7   num__Tenure WarehouseToHome                         4504 non-null   float64
 8   num__Tenure HourSpendOnApp                          4504 non-null   float64
 9

## Class imbalance

In [4]:
y_train.value_counts()

Churn
0    3746
1     758
Name: count, dtype: int64

In [5]:
# SMOTE for upsampling.

# Upsample by creating new, synthetic data points rather than duplicating existing ones.
# Its main advantages over naive methods like random oversampling include reduced overfitting, improved model generalization, and better preservation of minority class information
# !pip install imbalanced-learn

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

y_train_smote.value_counts()


c:\Users\Adithya\Desktop\bt4103\bt4103-grp9\.venv\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Churn
0    3746
1    3746
Name: count, dtype: int64

Data set for modelling is X_train_smote, X_test, y_train_smote and y_test


## Logistic Regression

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

logreg = LogisticRegression(solver='liblinear', max_iter=1000)

logreg.fit(X_train_smote, y_train_smote)

y_pred = logreg.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print("Accuracy:", accuracy)
print("Confusion Matrix:\n", conf_matrix)
print("Classification Report:\n", class_report)

Accuracy: 0.7992895204262878
Confusion Matrix:
 [[742 194]
 [ 32 158]]
Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.79      0.87       936
           1       0.45      0.83      0.58       190

    accuracy                           0.80      1126
   macro avg       0.70      0.81      0.73      1126
weighted avg       0.87      0.80      0.82      1126



## Random Forest

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import recall_score, accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import numpy as np

rf = RandomForestClassifier(random_state=42)

param_dist = {
    'n_estimators': [100, 200, 500, 800, 1000],
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],  # good for 76 features
    'bootstrap': [True, False]
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#Randomized because we have a relatively higher number of features
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,                  # number of random combinations to try
    scoring='accuracy',          # change to 'f1' or 'roc_auc' if needed
    cv=skf,
    verbose=1,
    random_state=42,
    n_jobs=-1                    # use all processors
)

random_search.fit(X_train_smote, y_train_smote)

print("Best Parameters:", random_search.best_params_)
print("Best CV Accuracy:", random_search.best_score_)

best_rf = random_search.best_estimator_
y_pred = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:,1]



print("\n=== Test Set Evaluation ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters: {'n_estimators': 800, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 50, 'bootstrap': False}
Best CV Accuracy: 0.9781101063370239

=== Test Set Evaluation ===
Accuracy: 0.9760213143872114
F1 Score: 0.928
ROC AUC: 0.9964434322986955
Confusion Matrix:
 [[925  11]
 [ 16 174]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.99      0.99       936
           1       0.94      0.92      0.93       190

    accuracy                           0.98      1126
   macro avg       0.96      0.95      0.96      1126
weighted avg       0.98      0.98      0.98      1126



In [8]:
w_rf = recall_score(y_test, y_pred)

## XGboost

In [9]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import numpy as np

xgb_clf = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

param_dist = {
    'n_estimators': [100, 200, 500, 800],
    'max_depth': [3, 5, 7, 10, 15],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2, 0.3, 0.5],
    'min_child_weight': [1, 3, 5, 7]
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_dist,
    n_iter=50,                  # number of random combinations to try
    scoring='accuracy',          # can change to 'f1' or 'roc_auc'
    cv=skf,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_smote, y_train_smote)

print("Best Parameters:", random_search.best_params_)
print("Best CV Accuracy:", random_search.best_score_)

best_xgb = random_search.best_estimator_
y_pred = best_xgb.predict(X_test)
y_proba = best_xgb.predict_proba(X_test)[:,1]

print("\n=== Test Set Evaluation ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters: {'subsample': 1.0, 'n_estimators': 800, 'min_child_weight': 3, 'max_depth': 15, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.8}
Best CV Accuracy: 0.9807791754360495

=== Test Set Evaluation ===
Accuracy: 0.9849023090586145
F1 Score: 0.9546666666666667
ROC AUC: 0.995968286099865
Confusion Matrix:
 [[930   6]
 [ 11 179]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99       936
           1       0.97      0.94      0.95       190

    accuracy                           0.98      1126
   macro avg       0.98      0.97      0.97      1126
weighted avg       0.98      0.98      0.98      1126



In [10]:
w_xgb = recall_score(y_test, y_pred)

## LightGBM

In [11]:
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

lgbm = lgb.LGBMClassifier(
    objective='binary',
    random_state=42,
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=5,
    min_child_samples=40,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2
)

param_dist = {
    'n_estimators': [100, 200, 500, 800],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [15, 31, 63, 127],
    'max_depth': [-1, 5, 10, 20],
    'min_child_samples': [10, 20, 30, 50],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_dist,
    n_iter=30,
    scoring='f1',
    cv=skf,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_smote, y_train_smote)

print("Best Parameters:", random_search.best_params_)
print("Best CV Score:", random_search.best_score_)

best_lgbm = random_search.best_estimator_
y_pred = best_lgbm.predict(X_test)
y_proba = best_lgbm.predict_proba(X_test)[:, 1]

print("\n=== Test Set Evaluation ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 3746, number of negative: 3746
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003676 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12234
[LightGBM] [Info] Number of data points in the train set: 7492, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

In [12]:
w_lgbm = recall_score(y_test, y_pred)

## Model ensemble

In [13]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import numpy as np

# -------------------------
# Get probabilities
# -------------------------
rf_proba = best_rf.predict_proba(X_test)[:, 1]
xgb_proba = best_xgb.predict_proba(X_test)[:, 1]
lgbm_proba = best_lgbm.predict_proba(X_test)[:, 1]

recalls = [w_rf, w_xgb, w_lgbm]
total = sum(recalls)

w_rf, w_xgb, w_lgbm = [r / total for r in recalls]

print("Weights:", w_rf, w_xgb, w_lgbm)

# -------------------------
# Weighted average
# -------------------------
y_proba = (w_rf * rf_proba +
           w_xgb * xgb_proba +
           w_lgbm * lgbm_proba)

# Convert to class (threshold = 0.5)
y_pred = (y_proba >= 0.5).astype(int)

# -------------------------
# Evaluation
# -------------------------
print("\n=== Test Set Evaluation (Weighted Ensemble) ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Weights: 0.3283018867924528 0.33773584905660375 0.3339622641509434

=== Test Set Evaluation (Weighted Ensemble) ===
Accuracy: 0.9866785079928952
F1 Score: 0.9597855227882037
ROC AUC: 0.9974808816914079
Confusion Matrix:
 [[932   4]
 [ 11 179]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99       936
           1       0.98      0.94      0.96       190

    accuracy                           0.99      1126
   macro avg       0.98      0.97      0.98      1126
weighted avg       0.99      0.99      0.99      1126



In [14]:
from sklearn.ensemble import VotingClassifier

voting_hard = VotingClassifier(
    estimators=[
        ("rf", best_rf),
        ("xgb", best_xgb),
        ("lgbm", best_lgbm)
    ],
    voting="hard"
)

voting_hard.fit(X_train, y_train)

y_pred = voting_hard.predict(X_test)

print("\n=== Test Set Evaluation (Hard Voting) ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 758, number of negative: 3746
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001889 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3809
[LightGBM] [Info] Number of data points in the train set: 4504, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.168295 -> initscore=-1.597760
[LightGBM] [Info] Start training from score -1.597760
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [15]:
import pandas as pd
import numpy as np

# Weighted ensemble scoring on X_test
rf_proba = best_rf.predict_proba(X_test)[:, 1]
xgb_proba = best_xgb.predict_proba(X_test)[:, 1]
lgbm_proba = best_lgbm.predict_proba(X_test)[:, 1]

ensemble_proba = (
    w_rf * rf_proba +
    w_xgb * xgb_proba +
    w_lgbm * lgbm_proba
)

ensemble_pred = (ensemble_proba >= 0.5).astype(int)

scored_test = X_test.copy()
scored_test["Churn_true"] = y_test.values
scored_test["Churn_pred"] = ensemble_pred
scored_test["Churn_probability"] = ensemble_proba

scored_test.to_csv("ensemble_test_scored.csv", index=False)
print("Saved: ensemble_test_scored.csv")

display(scored_test.head())

Saved: ensemble_test_scored.csv


,num__Tenure,num__NumberOfDeviceRegistered,num__SatisfactionScore,num__Complain,num__DaySinceLastOrder,num__CashbackAmount,num__Tenure CityTier,num__Tenure WarehouseToHome,num__Tenure HourSpendOnApp,num__Tenure NumberOfDeviceRegistered,...,num__DaySinceLastOrder CashbackAmount,cat__PreferredLoginDevice_Mobile Phone,cat__PreferedOrderCat_Laptop & Accessory,cat__PreferedOrderCat_Mobile,cat__PreferedOrderCat_Mobile Phone,cat__MaritalStatus_Married,cat__MaritalStatus_Single,Churn_true,Churn_pred,Churn_probability
0,0.822895,0.294877,-1.493573,-0.623912,0.997108,0.094099,1.993415,-0.028237,1.458203,0.915923,...,0.749874,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0.016013
1,1.900976,0.294877,-1.493573,1.602791,-0.409663,0.644226,3.545827,1.007027,1.838572,1.989424,...,-0.267522,1.0,0.0,0.0,0.0,1.0,0.0,0,0,0.004013
2,-0.614545,0.294877,-1.493573,-0.623912,1.278462,-0.170777,-0.651434,-0.464727,-0.367566,-0.515412,...,0.829646,1.0,1.0,0.0,0.0,0.0,1.0,0,0,0.002081
3,1.781189,0.294877,-0.771089,-0.623912,-0.691017,-0.924654,0.498501,0.234776,0.773540,1.870146,...,-0.713016,0.0,0.0,1.0,0.0,0.0,1.0,0,0,0.015907
4,-0.734332,-0.675976,-0.048604,1.602791,0.434400,0.440475,-0.248957,-0.705356,-0.824008,-0.753968,...,0.428333,0.0,0.0,0.0,0.0,1.0,0.0,0,0,0.055689


In [16]:
import shap

print("Computing SHAP-based ensemble feature importance...")

sample_size = min(2000, X_test.shape[0])
X_shap_sample = X_test.iloc[:sample_size].copy()
feature_names = X_test.columns.tolist()

# SHAP helper
def get_positive_class_shap(model, X):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    # Binary-class tree models may return:
    # - list of arrays
    # - 2D numpy array
    # - 3D numpy array
    if isinstance(shap_values, list):
        shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]
    elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        # shape could be (n_samples, n_features, n_classes)
        if shap_values.shape[2] > 1:
            shap_values = shap_values[:, :, 1]
        else:
            shap_values = shap_values[:, :, 0]

    return np.asarray(shap_values)

# Compute SHAP values for each model
rf_shap   = get_positive_class_shap(best_rf, X_shap_sample)
xgb_shap  = get_positive_class_shap(best_xgb, X_shap_sample)
lgbm_shap = get_positive_class_shap(best_lgbm, X_shap_sample)

# Mean absolute SHAP importance per model
rf_importance   = np.abs(rf_shap).mean(axis=0)
xgb_importance  = np.abs(xgb_shap).mean(axis=0)
lgbm_importance = np.abs(lgbm_shap).mean(axis=0)

# Weighted ensemble SHAP importance
ensemble_shap_importance = (
    w_rf * rf_importance +
    w_xgb * xgb_importance +
    w_lgbm * lgbm_importance
)

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "rf_shap_importance": rf_importance,
    "xgb_shap_importance": xgb_importance,
    "lgbm_shap_importance": lgbm_importance,
    "ensemble_shap_importance": ensemble_shap_importance
}).sort_values("ensemble_shap_importance", ascending=False).reset_index(drop=True)

feature_importance_df.to_csv("ensemble_shap_feature_importance.csv", index=False)
print("Saved: ensemble_shap_feature_importance.csv")

display(feature_importance_df.head(25))

c:\Users\Adithya\Desktop\bt4103\bt4103-grp9\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Computing SHAP-based ensemble feature importance...
Saved: ensemble_shap_feature_importance.csv


c:\Users\Adithya\Desktop\bt4103\bt4103-grp9\.venv\lib\site-packages\shap\explainers\_tree.py:586: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,feature,rf_shap_importance,xgb_shap_importance,lgbm_shap_importance,ensemble_shap_importance
0,num__Tenure,0.039271,1.121055,0.933997,0.703433
1,num__CityTier NumberOfAddress,0.020778,0.736683,0.716812,0.495014
2,cat__PreferedOrderCat_Laptop & Accessory,0.017565,0.637945,0.661644,0.442188
3,cat__MaritalStatus_Single,0.018035,0.583270,0.679159,0.429725
4,num__Tenure DaySinceLastOrder,0.022916,0.590982,0.626202,0.416247
5,num__SatisfactionScore Complain,0.014098,0.420220,0.403910,0.281443
6,num__Tenure OrderAmountHikeFromlastYear,0.032745,0.367760,0.401159,0.268928
7,num__Tenure SatisfactionScore,0.012817,0.416200,0.307250,0.247383
8,num__Tenure CityTier,0.017818,0.340700,0.292822,0.218708
9,num__NumberOfAddress Complain,0.019370,0.330372,0.255349,0.203215


In [17]:
import joblib
import json
from pathlib import Path

EXPORT_DIR = Path(".")

joblib.dump(pipeline, EXPORT_DIR / "ensemble_preprocessing_pipeline.joblib")
joblib.dump(best_rf, EXPORT_DIR / "ensemble_rf_model.joblib")
joblib.dump(best_xgb, EXPORT_DIR / "ensemble_xgb_model.joblib")
joblib.dump(best_lgbm, EXPORT_DIR / "ensemble_lgbm_model.joblib")

weights = {
    "w_rf": float(w_rf),
    "w_xgb": float(w_xgb),
    "w_lgbm": float(w_lgbm),
}

metadata = {
    "target": "Churn",
    "id_cols": ["CustomerID"],
    "raw_input_columns": [c for c in df.columns if c not in ["Churn"]],
    "selected_features": list(selected_features),
    "weights": weights,
}

with open(EXPORT_DIR / "ensemble_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)